# HW_L03_03_ML

## 1. Initial Plan and Preprocessing

In this exercise, you will use the Scikit-Learn library to build a complete machine learning pipeline. You will go beyond simple data analysis and build predictive models. The main focus is on data splitting, feature engineering, model selection, and hyperparameter tuning. In this exercise, you will use a synthetic dataset (`delivery_data.csv`) that simulates data from a shipping company, and you must train two machine learning models:

1. Regression model: To accurately estimate delivery time in minutes for display to the customer.
2. Classification model: To predict whether an order will be delivered with a delay (more than 45 minutes) or not (to send a discount code or a proactive apology).


In [86]:
# basic
import os

# Analysis
import numpy as np
import pandas as pd

# plot
import matplotlib.pyplot as plt
import seaborn as sns

## Part 1
Before training the model, we must prepare the data. For this purpose, the following steps must be followed:

1. Loading and Splitting:
   - Load the dataset.
   - Define the feature matrix (X) and the target vector (y). (Currently, use the `Delivery_Time_min` column for the target.)
   - Using `train_test_split`, split the data into training (80%) and testing (20%) sets. Be sure to use `random_state` so that results are reproducible.

2. Building the Transformation Pipeline

- We have two types of features: numerical features (`Distance_km`, etc.) and categorical features (`Traffic_Level`, `Weather`).
- Build a `ColumnTransformer` that performs the following operations:
  - For numerical columns: fill missing values with the mean (`SimpleImputer`) and then standardize the data (`StandardScaler`).
  - For categorical columns: fill missing values with the most frequent value (`most_frequent`) and then convert text into vectors (`OneHotEncoder`).
- Note: Linear models require the data to be scaled and cannot process string values directly.



In [87]:
# Read data
delivery_time_min = pd.read_csv("delivery_data.csv")

In [88]:
delivery_time_min.columns

Index(['Distance_km', 'Prep_Time_min', 'Courier_Experience_yrs', 'Weather',
       'Traffic_Level', 'Vehicle_Type', 'Delivery_Time_min', 'Is_Late'],
      dtype='str')

In [89]:
delivery_time_min.head()

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min,Is_Late
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542,0
1,4.723471,14.277407,2,Sunny,High,Car,35.527712,0
2,6.295377,11.037900,9,Rainy,High,Scooter,48.567934,1
3,8.046060,13.460192,7,Sunny,High,Car,50.452710,1
4,4.531693,5.531927,7,Rainy,Medium,Bicycle,47.632817,1


In [90]:
delivery_time_min.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Distance_km             2000 non-null   float64
 1   Prep_Time_min           1895 non-null   float64
 2   Courier_Experience_yrs  2000 non-null   int64  
 3   Weather                 1893 non-null   str    
 4   Traffic_Level           2000 non-null   str    
 5   Vehicle_Type            2000 non-null   str    
 6   Delivery_Time_min       2000 non-null   float64
 7   Is_Late                 2000 non-null   int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 125.1 KB


In [91]:
delivery_time_min.describe()

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Delivery_Time_min,Is_Late
count,2000.000000,1895.000000,2000.000000,2000.000000,2000.000000
mean,5.097968,15.031132,4.552500,53.317833,0.549000
std,1.956540,4.942245,2.885898,27.200221,0.497718
min,0.500000,5.000000,0.000000,10.000000,0.000000
25%,3.754676,11.453845,2.000000,33.683481,0.000000
50%,5.089383,15.033999,5.000000,47.682737,1.000000
75%,6.365955,18.350682,7.000000,67.152951,1.000000
max,12.705463,34.631189,9.000000,183.617375,1.000000


In [92]:
delivery_time_min.isnull().sum()

Distance_km                 0
Prep_Time_min             105
Courier_Experience_yrs      0
Weather                   107
Traffic_Level               0
Vehicle_Type                0
Delivery_Time_min           0
Is_Late                     0
dtype: int64

The two columns `Prep_Time_min` and `Weather` contain NaN values, so we fill them with the mean (`SimpleImputer`) and then standardize the data (`StandardScaler`).


In [93]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


# drop Is_Late columns 
data_frame_delivery = delivery_time_min.drop(columns=["Is_Late"]).copy()

# 1st part: create x and y
# label columns or target
target = "Delivery_Time_min"

# create X and y
# X: model learn - X is matrix so X is Capital character
X = data_frame_delivery.drop(columns=[target])
# y: Answer or label - y is vector and Small character
y = data_frame_delivery[target]


# 2nd part: train_test_split
# user train_test_split for creat train data and test data
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.2,
                                                    random_state = 1)

# 3rd part: ColumnTransformer
# numeric columns and categorical columns
numerical_columns = ["Distance_km", "Prep_Time_min", "Courier_Experience_yrs"]
categorical_columns = ["Weather", "Traffic_Level", "Vehicle_Type"] # this is str

# pipeline for numerical columns
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="mean")),
                                  ("scaler", StandardScaler())])

categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("one_hot", OneHotEncoder(handle_unknown="ignore"))])


preprocessor = ColumnTransformer(
    transformers=[("num", numerical_transformer, numerical_columns),
                  ("cat", categorical_transformer, categorical_columns)])

In [94]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

test data by preprocessor

In [95]:
data_frame_delivery.head(2)

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542
1,4.723471,14.277407,2,Sunny,High,Car,35.527712


In [96]:
pd.DataFrame(preprocessor.fit_transform(X_train))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.450928,-0.713998,0.854953,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.216522,0.477746,-1.221856,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.382447,0.594076,-0.875721,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1.508346,-1.411686,-1.567990,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.432828,-0.062792,-0.183451,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,0.061359,-0.443400,-0.875721,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1596,0.024217,1.054088,-0.875721,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1597,-0.530961,0.895269,1.547222,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1598,0.590966,0.733857,0.508818,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


validation columns

In [97]:

len_categorical_columns = 0
for i in categorical_columns:
    len_categorical_columns += len((data_frame_delivery[i].unique().dropna()))

print(f"len categorical columns : {len_categorical_columns}")
print(f"len numerical columns : {len(numerical_columns)}")
print(f"len total columns : {len_categorical_columns + len(numerical_columns)}")
print(f"len preprocessor columns : {pd.DataFrame(preprocessor.fit_transform(X_train)).columns.stop}")



len categorical columns : 11
len numerical columns : 3
len total columns : 14
len preprocessor columns : 14


## Part 2: Delivery Time Prediction 

In this section, you must train a model on the data that can estimate the delivery time. The implementation steps are as follows:

### 1. Basic Linear Regression

- Build a `Pipeline` that includes the `ColumnTransformer` from the previous step and the `LinearRegression` model.
- Train the model on the training data (`X_train`, `y_train`).
- Make predictions on the test data (`X_test`).
- Calculate the `MAE` (Mean Absolute Error) and `R² Score` metrics. On average, how many minutes of error does your prediction have?

### 2. Bias-Variance 

- The relationship between traffic and delivery time is not necessarily linear.
- Build a new pipeline that adds polynomial features (`PolynomialFeatures`) with degree 2 before the regression step.
- Train and evaluate the model. Did the `R²` score improve? Has the model suffered from overfitting?

### 3. Regularization (Ridge/Lasso)

- Using polynomial features often causes overfitting.
- Replace the `LinearRegression` model with `Ridge` (L2 regularization).
- Use `GridSearchCV` to find the best `alpha` value among `[0.1, 1.0, 10.0]`.
- Print the best parameter and the best score.


In [115]:
#1. Basic Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#---------------------------------
# 1st create model LinearRegression 
model_LinearRegression = Pipeline([("preprocessor", preprocessor),
                                   ("regressor", LinearRegression())])

#---------------------------------
# 2nd fit train data
model_LinearRegression.fit(X_train, y_train)

#---------------------------------
# 3rd predict data
y_predict = model_LinearRegression.predict(X_test)

#---------------------------------
# 4th Calculate the evaluation metrics MAE and R2_score

# MAE
mae_linearregresseion = mean_absolute_error(y_test, y_predict)

# MSE
mse_linearregresseion = mean_squared_error(y_test, y_predict)

# R² Score
r2_score_linearregresseion = r2_score(y_test, y_predict)

print(f"MAE of LinearRegression is {mae_linearregresseion} min")
print(f"MSE of LinearRegression is {mse_linearregresseion}")
print(f"Square MSE of LinearRegression is {mse_linearregresseion**(0.5)} min")
print(f"R2_score of LinearRegression is {r2_score_linearregresseion}")



MAE of LinearRegression is 6.797055213307144 min
MSE of LinearRegression is 76.93314928533395
Square MSE of LinearRegression is 8.771154387270467 min
R2_score of LinearRegression is 0.8767002020076216


According to the MAE: the model has an error of 6.797 minutes.

According to the MSE: this parameter squares the errors and then takes their average. This error metric is sensitive to outliers and penalizes them more heavily. The square root of MSE means that the model has an approximate typical error of about 8.77 minutes.

According to the R² Score: the model was able to learn correctly up to 87.67%.


In [116]:
# 2. Bias-Variance

from sklearn.preprocessing import PolynomialFeatures